# 07 – Bonus: small ML classifier (TF-IDF + logistic regression) vs the LLM judge

Goal: train a small classifier on the 60 labelled explanations and compare it with the LLM judge on accuracy, cost and speed.

## Approach
- Model: TF-IDF on words and word pairs, then logistic regression (`C=10`, not tuned). It only predicts the verdict; it cannot write feedback.
- Accuracy is measured with 5-fold cross-validation, repeated with 5 different shuffles because 60 rows give very noisy numbers.
- **Two versions of cross-validation.** The 60 labelled rows contain only 52 distinct texts, so the same text can land in both the training and the test fold, which makes the score look better. "Grouped" cross-validation keeps identical texts together in the same fold and is the honest number.
- Speed and cost are measured on the same machine; LLM figures come from notebooks 05 and 06 and the pricing page.
- As a second check, the classifier (trained on the 60 rows) is compared with the LLM verdicts on the 203 other explanations the LLM judged. This is agreement with the LLM, not accuracy: those rows have no human labels.

In [1]:
#imports and load the 60 labelled explanations
import json
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

pd.set_option("display.max_colwidth", None)
expl = pd.read_csv("../data/clean/explanations_clean.csv")
lab = expl[expl.label.notna()].reset_index(drop=True)
X, y = lab.explanation_text, lab.label
print(len(lab), "labelled rows;", X.nunique(), "distinct texts")

def make_model():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                         LogisticRegression(max_iter=1000, C=10))

60 labelled rows; 52 distinct texts


In [2]:
#cross-validated accuracy: plain vs grouped (identical texts kept in the same fold), 5 shuffles each
groups = X.factorize()[0]
plain, grouped = [], []
for seed in range(5):
    p = cross_val_predict(make_model(), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed))
    g = cross_val_predict(make_model(), X, y, cv=StratifiedGroupKFold(5, shuffle=True, random_state=seed), groups=groups)
    plain.append(accuracy_score(y, p))
    grouped.append(accuracy_score(y, g))
pd.DataFrame({"plain CV (inflated by duplicate texts)": plain, "grouped CV (honest)": grouped}).round(2).T.assign(mean=lambda d: d.mean(axis=1).round(3))

,0,1,2,3,4,mean
plain CV (inflated by duplicate texts),0.82,0.82,0.77,0.75,0.83,0.798
grouped CV (honest),0.80,0.78,0.70,0.58,0.72,0.716


In [3]:
#speed: train on all 60 rows, then predict all 724 explanations
t = time.time(); model = make_model().fit(X, y); fit_s = time.time() - t
t = time.time(); model.predict(expl.explanation_text); pred_s = time.time() - t
print(f"training: {fit_s:.3f} s | predicting 724 explanations: {pred_s:.3f} s ({pred_s / 724 * 1000:.3f} ms each)")

training: 0.024 s | predicting 724 explanations: 0.037 s (0.052 ms each)


In [4]:
#agreement with the LLM (prompt v2) on explanations the classifier was not trained on
llm = {}
for line in open("../outputs/judgments_v2.jsonl", encoding="utf-8"):
    if line.strip():
        r = json.loads(line)
        llm[r["explanation_id"]] = r["verdict"]
rest = expl[expl.label.isna() & expl.explanation_id.isin(llm)].copy()
rest["llm"] = rest.explanation_id.map(llm)
rest["clf"] = model.predict(rest.explanation_text)
seen = set(lab.explanation_text)
rest["text_in_labelled_set"] = rest.explanation_text.isin(seen)
print("explanations compared:", len(rest), "| agreement with the LLM:", round((rest.llm == rest.clf).mean(), 3))
print(rest.groupby("text_in_labelled_set").apply(lambda d: pd.Series({"rows": len(d), "agreement": round((d.llm == d.clf).mean(), 3)})))
print(pd.crosstab(rest.llm, rest.clf, rownames=["LLM"], colnames=["classifier"]))
rest[(~rest.text_in_labelled_set) & (rest.llm != rest.clf)][["explanation_id", "explanation_text", "llm", "clf"]].head(6)

explanations compared: 203 | agreement with the LLM: 0.897
                       rows  agreement
text_in_labelled_set                  
False                 149.0      0.859
True                   54.0      1.000
classifier  partial  understood  wrong
LLM                                   
partial          78           3      1
understood        3          72      0
wrong            14           0     32


,explanation_id,explanation_text,llm,clf
2,E0003,"Plants eat sunlight directly, sunlight is their food. hope this is right!",wrong,partial
26,E0027,It is the process where plants turn oxygen into water using the sun's heat.,wrong,partial
27,E0028,"Plants eat sunlight directly, sunlight is their food. (sorry for typos)",wrong,partial
100,E0101,"Plants eat sunlight directly, sunlight is their food.",wrong,partial
122,E0123,"Plants eat sunlight directly, sunlight is their food. i think",wrong,partial
173,E0174,"When supply goes up, prices also go up because companies earn more. i think",wrong,partial


## Results and findings

| | Accuracy on the 60 labelled rows | Cost per call | Time per explanation | Writes feedback |
|---|---|---|---|---|
| TF-IDF + logistic regression | 0.80 plain CV, **0.72 grouped CV** (mean of 5 shuffles; ranged 0.58 to 0.80) | none | about 0.05 ms | no |
| LLM judge, prompt v1 | 0.73 | about $0.00036 | about 1.3 s | yes |
| LLM judge, prompt v2 | 1.00 (tuned on the same 60 rows, so optimistic) | about $0.00036 | about 1.3 s | yes |

LLM cost: about 590 input and 70 output tokens per call at Gemini 3.5 Flash-Lite paid prices of $0.30 and $2.50 per 1M tokens (Google pricing page, 22 Oct 2026). Time per call was measured on 5 live calls.

- **A tiny classifier gets surprisingly far on this data.** Grouped CV accuracy of about 0.72 is close to the first LLM prompt (0.73), costs nothing and runs in microseconds. It also agrees with the LLM on about 90% of the other explanations.
- **But the numbers are fragile.** 60 rows is very small: the grouped score changed from 0.58 to 0.80 just by reshuffling. The plain score (0.80) is flattered by duplicate texts. The agreement is also inflated, because 54 of the 203 comparison rows have exactly the same text as a training row (agreement 100% there, about 86% on the 149 rows with new text).
- **Where it fails:** the classifier labels many LLM-`wrong` answers as `partial` (14 of 46), because wrong and partial answers share the same words (sunlight, plants, price). It matches words, not meaning, so a new wording or a contradiction is easy to miss. It cannot tell "plants get food from the soil" from a partly right answer reliably.
- **It cannot do the actual job.** The task is feedback for the student (one thing right, one thing missed, a hint). A classifier only outputs a label.
- **One lucky case, not evidence:** it labelled the Hinglish explanation (E0691) `understood`, like the LLM, probably because it contains words such as chlorophyll, CO2 and glucose. I would not expect this to hold for Tamil or Kannada text.
- **Honest limits:** agreement with the LLM is not accuracy (the LLM is not ground truth), `C` was not tuned, and results come from one small, repetitive dataset.